# Experiments 33-40
Pruebas de validación ajustando Confidence y IOU setup para Non-Maximum Suppression (NMS).

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Weights:** Exp. 26 *(Full fine-tuned, no freeze)*
- **Experiments:**
    1. Both lower: `conf=0.20` | `iou=0.4`
    1. Both higher: `conf=0.30` | `iou=0.75`
    1. Even lower Confidence: `conf=0.15`
    1. Even lower Confidence + low IOU: `conf=0.15` | `iou=0.4`
    1. Even lower IOU: `iou=0.3`
    1. Both even lower: `conf=0.15` | `iou=0.3`
- **Reference:** Default parameters: `conf=0.25` | `iou=0.6`

| Parameter | Type  | Value | Description                                                                                                                                                                                             |
|-----------|-------|-------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `conf`    | float | 0.001 | Sets the minimum confidence threshold for detections. Lower values increase recall but may introduce more false positives. Used during **validation** to compute precision-recall curves. |
| `iou`     | float | 0.6   | Sets the **Intersection Over Union** threshold for **Non-Maximum Suppression**. Controls duplicate detection elimination.                                                                       |

## Init

In [1]:
import os
import shutil
import fnmatch
import pickle

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.5/974.5 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 70.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

## Helper Functions

In [131]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [132]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [133]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [134]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [135]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [136]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Graph functions

In [137]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [138]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [139]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)


# Datasets builder

## Importing from Drive

In [9]:
!rm -rf /content/sample_data

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px	      best_e26.pt  models
3.5m.v3i.yolov8.640px.aug.v1  Inference    runs


In [12]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 6 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 'runs',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'best_e26.pt']

In [13]:
choose_dataset = 1
index = choose_dataset - 1
model = os.listdir(drive_path)[index]
print("Chosen model:", model)

Chosen model: 3.5m.v3i.yolov8.640px


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [14]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path
src_folder = f"/content/YOLO/{model}"

## Download model

In [15]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [16]:
# Load stored model (Exp. 26)
model = YOLO("/content/drive/MyDrive/YOLO/best_e26.pt")

# Finetuning

In [112]:
# Libera memoria de la GPU en caso de OOM error
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

0

### Info

In [ ]:
!nvidia-smi

Sun Apr 13 01:18:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!yolo version

8.3.107


In [ ]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

-----
## Reference (idem 26)
### *Full fine-tuned (no freeze) | Defaul values*
`conf = 0.25` | `iou = 0.6`

### Validation

In [182]:
# Validate the model
results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=0.25, # default value
          iou=0.6, # default value
          verbose=True,
          save_json=True)

Ultralytics 8.3.107 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.42s/it]


                   all        108       2409      0.573      0.545      0.536      0.204
Speed: 5.9ms preprocess, 21.5ms inference, 0.0ms loss, 2.2ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


### Metrics

In [183]:
gimme_metrics(results)

Total objects detected: 3325.0
Confusion matrix:
['43.58%', '27.55%']
['28.87%', '0.00%']


In [184]:
save_json(results)

✅ JSON file stored in: runs/detect/val


-----
## Experiment 33
### *Full fine-tuned (no freeze) | Lower confidence*
`conf = 0.20`

### Validation

In [185]:
# Validate the model
results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=0.20,
          iou=0.6, # default value
          verbose=True,
          save_json=True)

Ultralytics 8.3.107 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.25s/it]


                   all        108       2409      0.573      0.545      0.537      0.203
Speed: 6.1ms preprocess, 21.4ms inference, 0.0ms loss, 1.3ms postprocess per image
Saving runs/detect/val2/predictions.json...
Results saved to runs/detect/val2


### Metrics

In [186]:
gimme_metrics(results)

Total objects detected: 3444.0
Confusion matrix:
['43.82%', '30.05%']
['26.13%', '0.00%']


In [187]:
save_json(results)

✅ JSON file stored in: runs/detect/val2


-----
## Experiment 34
### *Full fine-tuned (no freeze) | Lower IOU*
`iou = 0.4`

### Validation

In [188]:
# Validate the model
results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=0.25, # default value
          iou=0.4,
          verbose=True,
          save_json=True)

Ultralytics 8.3.107 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.19s/it]


                   all        108       2409      0.581      0.533      0.537      0.204
Speed: 3.7ms preprocess, 21.5ms inference, 0.0ms loss, 2.4ms postprocess per image
Saving runs/detect/val3/predictions.json...
Results saved to runs/detect/val3


### Metrics

In [189]:
gimme_metrics(results)

Total objects detected: 3249.0
Confusion matrix:
['43.74%', '25.85%']
['30.41%', '0.00%']


In [190]:
save_json(results)

✅ JSON file stored in: runs/detect/val3


-----
## Experiment 35
### *Full fine-tuned (no freeze) | Both lower*
`conf = 0.20` | `iou = 0.4`

### Validation

In [191]:
# Validate the model
results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=0.20,
          iou=0.4,
          verbose=True,
          save_json=True)

Ultralytics 8.3.107 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.14s/it]


                   all        108       2409      0.581      0.533      0.537      0.203
Speed: 6.4ms preprocess, 21.6ms inference, 0.0ms loss, 1.8ms postprocess per image
Saving runs/detect/val4/predictions.json...
Results saved to runs/detect/val4


### Metrics

In [192]:
gimme_metrics(results)

Total objects detected: 3339.0
Confusion matrix:
['44.00%', '27.85%']
['28.15%', '0.00%']


In [193]:
save_json(results)

✅ JSON file stored in: runs/detect/val4


-----
## Experiment 36
### *Full fine-tuned (no freeze) | Both higher*
`conf = 0.30` | `iou = 0.75`

### Validation

In [194]:
# Validate the model
results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=0.30,
          iou=0.75,
          verbose=True,
          save_json=True)

Ultralytics 8.3.107 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.03s/it]


                   all        108       2409      0.563      0.525      0.528      0.202
Speed: 4.5ms preprocess, 21.5ms inference, 0.0ms loss, 1.4ms postprocess per image
Saving runs/detect/val5/predictions.json...
Results saved to runs/detect/val5


### Metrics

In [195]:
gimme_metrics(results)

Total objects detected: 3393.0
Confusion matrix:
['41.85%', '29.00%']
['29.15%', '0.00%']


In [196]:
save_json(results)

✅ JSON file stored in: runs/detect/val5


-----
## Experiment 37
### *Full fine-tuned (no freeze) | Even lower confidence*
`conf = 0.15`

### Validation

In [197]:
# Validate the model
results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=0.15,
          iou=0.6, # default value
          verbose=True,
          save_json=True)

Ultralytics 8.3.107 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.54s/it]


                   all        108       2409      0.573      0.545      0.536      0.199
Speed: 6.7ms preprocess, 21.8ms inference, 0.0ms loss, 2.5ms postprocess per image
Saving runs/detect/val6/predictions.json...
Results saved to runs/detect/val6


### Metrics

In [198]:
gimme_metrics(results)

Total objects detected: 3630.0
Confusion matrix:
['42.92%', '33.64%']
['23.44%', '0.00%']


In [199]:
save_json(results)

✅ JSON file stored in: runs/detect/val6


-----
## Experiment 38
### *Full fine-tuned (no freeze) | Even lower confidence + low IOU*
`conf = 0.15` | `iou = 0.4`

### Validation

In [202]:
# Validate the model
results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=0.15,
          iou=0.6, # default value
          verbose=True,
          save_json=True)

Ultralytics 8.3.107 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.06s/it]


                   all        108       2409      0.573      0.545      0.536      0.199
Speed: 4.3ms preprocess, 21.1ms inference, 0.0ms loss, 2.3ms postprocess per image
Saving runs/detect/val7/predictions.json...
Results saved to runs/detect/val7


### Metrics

In [203]:
gimme_metrics(results)

Total objects detected: 3630.0
Confusion matrix:
['42.92%', '33.64%']
['23.44%', '0.00%']


In [204]:
save_json(results)

✅ JSON file stored in: runs/detect/val7


-----
## Experiment 39
### *Full fine-tuned (no freeze) | Even lower IOU*
`iou = 0.3`

### Validation

In [205]:
# Validate the model
results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=0.25, # default value
          iou=0.3,
          verbose=True,
          save_json=True)

Ultralytics 8.3.107 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.42s/it]


                   all        108       2409      0.578       0.53      0.535      0.205
Speed: 7.8ms preprocess, 21.6ms inference, 0.0ms loss, 2.5ms postprocess per image
Saving runs/detect/val8/predictions.json...
Results saved to runs/detect/val8


### Metrics

In [206]:
gimme_metrics(results)

Total objects detected: 3218.0
Confusion matrix:
['43.51%', '25.14%']
['31.35%', '0.00%']


In [207]:
save_json(results)

✅ JSON file stored in: runs/detect/val8


-----
## Experiment 40
### *Full fine-tuned (no freeze) | Both even lower*
`conf = 0.15` | `iou = 0.3`

### Validation

In [208]:
# Validate the model
results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=0.15,
          iou=0.3,
          verbose=True,
          save_json=True)

Ultralytics 8.3.107 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.03s/it]


                   all        108       2409      0.581      0.528      0.534      0.201
Speed: 4.5ms preprocess, 21.4ms inference, 0.0ms loss, 1.8ms postprocess per image
Saving runs/detect/val9/predictions.json...
Results saved to runs/detect/val9


### Metrics

In [209]:
gimme_metrics(results)

Total objects detected: 3384.0
Confusion matrix:
['43.35%', '28.81%']
['27.84%', '0.00%']


In [210]:
save_json(results)

✅ JSON file stored in: runs/detect/val9


----

## Save all results

In [211]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save/
